In [1]:
import lfox
import lfox.lattice as lat
import lfox.evolution.hmc as lhmc
import jax
import jax.numpy as jnp
import numpy as np

from functools import partial

# Imports below require "dev" environment
import matplotlib.pyplot as plt
import lsqfit
import gvar as gv
import tqdm

# Double precision!
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_threefry_partitionable", True)

In [2]:
# Define the action
class ScalarAction(lhmc.Action):        

    @staticmethod
    @jax.jit
    def _S(fields, params):
        phi = fields[0]
        S = phi**2
        
        for ax in range(phi.d):
            S -= 2 * params['kappa'] * phi * phi.nn_field(ax)

        S += params['lambda'] * (phi**2 - 1)**2
        return S
    
    # Exact force function instead of autodiff, for testing purposes
    @staticmethod
    @jax.jit
    def exact_force(fields, params):
        phi = fields[0]

        J = phi.nn_field(axis=0, shift=1) + phi.nn_field(axis=0, shift=-1)
        for ax in range(1,phi.d):
            J += phi.nn_field(axis=ax, shift=1)
            J += phi.nn_field(axis=ax, shift=-1)

        F = -2 * params['kappa'] * J
        F += 2 * phi.F
        F += 4 * params['lambda'] * (phi.F**2 - 1) * phi.F

        return F

# Component actions for testing composition
class ScalarKineticAction(lhmc.Action):
    @staticmethod
    @jax.jit
    def _S(fields, params):
        phi = fields[0]
        S = phi**2

        for ax in range(phi.d):
            S -= 2 * params['kappa'] * phi * phi.nn_field(ax)

        return S

class ScalarQuarticInt(lhmc.Action):    
    @staticmethod
    @jax.jit
    def _S(fields, params):
        phi = fields[0]
        return params['lambda'] * (phi**2 - 1)**2




In [3]:
# Actions depend on parameters, fields are arbitrary including dimension
S_A = ScalarAction(params = {'kappa': 0.18, 'lambda': 1.0}, field_names=['phi'])
S_B = ScalarAction(params = {'kappa': 0.17, 'lambda': 1.0}, field_names=['phi'])

# Force calculation should work automatically, without explicit definition above
#F_A = S_A.get_forces()
#print(F_A)

# Calling the action object as a function, using fields as input, should call and return the summed action functional
L4 = lat.SquareLattice(dims=(4,4))
L6 = lat.SquareLattice(dims=(6,6))
phi4 = lat.LatticeField(L4).unit_fill()
phi6 = lat.LatticeField(L6).unit_fill()
SA_total_4 = S_A.S({'phi': phi4})
SA_total_6 = S_A.S({'phi': phi6})
SB_total_4 = S_B.S({'phi': phi4})

print(SA_total_4, SA_total_6, SB_total_4)

print(S_A.dS({'phi': phi4})['phi'].F)

4.48 10.08 5.119999999999998
[[0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]]


I0000 00:00:1703800754.086044       1 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.


In [4]:
S_A.S({'phi': phi4, 'phi2': phi4})

Array(4.48, dtype=float64)

In [5]:
%timeit S_A.S({'phi': phi4})
%timeit S_A._S(fields=[phi4], params=S_A.params)

9.2 µs ± 27.9 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
7.81 µs ± 18.4 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [7]:
%%prun

for _ in range(10000):
    S_A.S({'phi': phi4})
#    S_A._S(fields=[phi4], params=S_A.params)
#    S_A.S_field(fields={'phi': phi4})

         310003 function calls in 0.181 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.086    0.086    0.181    0.181 <string>:1(<module>)
    10000    0.051    0.000    0.087    0.000 _module.py:687(_flatten_module)
    10000    0.012    0.000    0.020    0.000 dataclasses.py:1233(fields)
   120000    0.007    0.000    0.007    0.000 {method 'append' of 'list' objects}
    40000    0.006    0.000    0.006    0.000 dataclasses.py:1248(<genexpr>)
    60000    0.005    0.000    0.005    0.000 {built-in method builtins.getattr}
    10000    0.005    0.000    0.005    0.000 <string>:2(__eq__)
    10000    0.003    0.000    0.003    0.000 lattice.py:154(_tree_flatten)
    30000    0.003    0.000    0.003    0.000 {method 'get' of 'mappingproxy' objects}
    10000    0.002    0.000    0.002    0.000 <string>:2(__init__)
    10000    0.001    0.000    0.001    0.000 {method 'values' of 'dict' objects}
        1    

In [8]:
print(S_A.dS({'phi': phi4})['phi'].F)
print(S_A.exact_force(fields=[phi4], params=S_A.params).F)

%timeit S_A.dS({'phi': phi4})
%timeit S_A.exact_force(fields=[phi4], params=S_A.params)

[[0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]]
[[0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]]
11.1 µs ± 37.1 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
7.85 µs ± 34.6 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [15]:
# Try a bigger field to test class overhead

Lbig = lat.SquareLattice(dims=(256,256))
phibig = lat.LatticeField(Lbig).unit_fill()

%timeit S_A.dS({'phi': phibig})
%timeit S_A.exact_force(fields=[phibig], params=S_A.params)

109 µs ± 37.3 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
78.1 µs ± 581 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [25]:
%%prun 

for _ in range(10000):
    S_A.dS({'phi': phibig})
#    S_A.exact_force(fields=[phibig], params=S_A.params)


         370003 function calls in 1.224 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    1.074    1.074    1.224    1.224 <string>:1(<module>)
    10000    0.056    0.000    0.096    0.000 _module.py:687(_flatten_module)
    10000    0.024    0.000    0.026    0.000 lattice.py:118(__init__)
    30000    0.015    0.000    0.015    0.000 <string>:2(__eq__)
    10000    0.014    0.000    0.023    0.000 dataclasses.py:1233(fields)
    10000    0.009    0.000    0.035    0.000 lattice.py:164(_tree_unflatten)
    40000    0.007    0.000    0.007    0.000 dataclasses.py:1248(<genexpr>)
   120000    0.007    0.000    0.007    0.000 {method 'append' of 'list' objects}
    60000    0.006    0.000    0.006    0.000 {built-in method builtins.getattr}
    10000    0.004    0.000    0.004    0.000 lattice.py:154(_tree_flatten)
    30000    0.003    0.000    0.003    0.000 {method 'get' of 'mappingproxy' objects}
    10000    

In [6]:
# We should be able to build a joint action with *shared* fields, by composition.
S_A_kin = ScalarKineticAction(params={'kappa': 0.18}, field_names=['chi'])
S_A_int = ScalarQuarticInt(params={'lambda': 1.0}, field_names=['chi'])

S_A_joint = S_A_kin + S_A_int
print(S_A_joint.S({'chi': phi4}))  # Should match cell above
      
# On the other hand, we should also be able to compose multiple copies of a given action,
# without making the fields shared.
S_A_different = ScalarAction(params={'kappa': 0.18, 'lambda': 1.0}, field_names=['phi2'])
S_A_disjoint = S_A + S_A_different

print(S_A_disjoint.S(fields={'phi':phi4, 'phi2': phi4}))  # Should be 2x the action above

4.48
8.96


In [7]:
# Some edge cases/error modes:

# Trying to combine actions with two different field dimensions should error
try:
    print(S_A_disjoint.S(fields={'phi': phi4, 'phi2': phi6}))
except Exception as e:
    print(e)


add got incompatible shapes for broadcasting: (4, 4), (6, 6).


In [14]:
# Trying to see if mutation is a failure mode for:
# Actions
print(S_A.params)
print(S_A.S({'phi': phi4}))
F = S_A.get_forces()
print(F({'phi': phi4})['phi'].F)
print(S_A.dS({'phi': phi4})['phi'].F)
S_A.params['kappa'] = 0.01
print(S_A.params)
print(S_A.S({'phi': phi4}))
F = S_A.get_forces()
print(F({'phi': phi4})['phi'].F)
print(S_A.dS({'phi': phi4})['phi'].F)

{'kappa': 0.18, 'lambda': 1.0}
4.48
[[0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]]
[[0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]]
{'kappa': 0.01, 'lambda': 1.0}
15.360000000000001
[[0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]]
[[0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]]


In [9]:
S_A.dS({'phi': phi4})['phi'].F

Array([[1.92, 1.92, 1.92, 1.92],
       [1.92, 1.92, 1.92, 1.92],
       [1.92, 1.92, 1.92, 1.92],
       [1.92, 1.92, 1.92, 1.92]], dtype=float64)

In [11]:
%timeit F({'phi': phi4})['phi'].F
%timeit S_A.dS({'phi': phi4})['phi'].F

5.53 µs ± 5.78 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
5.73 µs ± 169 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
